In [ ]:
import numpy as np
import einops
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from lerobot.datasets.lerobot_dataset import HF_LEROBOT_HOME
from lerobot.datasets.lerobot_dataset import LeRobotDataset

repo_id = "flow929/ledataset_libero_10"
root = HF_LEROBOT_HOME / repo_id


def parse_image(image):
    image = np.asarray(image)
    if np.issubdtype(image.dtype, np.floating):
        image = (255 * image).astype(np.uint8)
    if image.shape[0] == 3:
        image = einops.rearrange(image, "c h w -> h w c")
    return image

class MyPiDataset(Dataset):
    def __init__(self, base_ds):
        self.base_ds = base_ds

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        sample = self.base_ds[idx]

        out = {
            "state": sample["state"],
            "image": {
                "base_0_rgb": parse_image(sample["image"]),
                "left_wrist_0_rgb": parse_image(sample["wrist_image"]),
                "right_wrist_0_rgb": np.zeros_like(parse_image(sample["image"])),
            },
            "image_mask": {
                "base_0_rgb": True,
                "left_wrist_0_rgb": True,
                "right_wrist_0_rgb": False,
            },
            "actions": sample["actions"],
            "prompt": sample["task"],
        }
        return out
    

print("dataset root:", root)
print("exists:", root.exists())

for p in sorted(root.iterdir()):
    print(p.name)

ds = LeRobotDataset(repo_id)
my_ds = MyPiDataset(ds)

loader = DataLoader(
    my_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(loader))
print(batch.keys())
print(batch["state"].shape)
print(batch["actions"].shape)

print(batch["image"]["base_0_rgb"].shape)
print(batch["image"]["left_wrist_0_rgb"].shape)
print(batch["image"]["right_wrist_0_rgb"].shape)

print(batch["image_mask"]["base_0_rgb"])
print(batch["image_mask"]["left_wrist_0_rgb"])
print(batch["image_mask"]["right_wrist_0_rgb"])
print(batch["prompt"])

dataset root: /home/flow/.cache/huggingface/lerobot/flow929/ledataset_libero_10
exists: True
data
images
meta
dict_keys(['state', 'image', 'image_mask', 'actions', 'prompt'])
torch.Size([4, 8])
torch.Size([4, 7])
torch.Size([4, 256, 256, 3])
torch.Size([4, 256, 256, 3])
torch.Size([4, 256, 256, 3])
tensor([True, True, True, True])
tensor([True, True, True, True])
tensor([False, False, False, False])


In [5]:
sample = ds[0]
sample

{'image': tensor([[[0.7843, 0.7843, 0.7804,  ..., 0.7765, 0.7725, 0.7725],
          [0.7843, 0.7804, 0.7804,  ..., 0.7765, 0.7725, 0.7725],
          [0.7804, 0.7765, 0.7804,  ..., 0.7725, 0.7725, 0.7725],
          ...,
          [0.0784, 0.0745, 0.0667,  ..., 0.0667, 0.0667, 0.0667],
          [0.1451, 0.1412, 0.1373,  ..., 0.0314, 0.0314, 0.0314],
          [0.1804, 0.1765, 0.1725,  ..., 0.0118, 0.0118, 0.0118]],
 
         [[0.7137, 0.7137, 0.7098,  ..., 0.7098, 0.7059, 0.7059],
          [0.7137, 0.7098, 0.7098,  ..., 0.7098, 0.7059, 0.7059],
          [0.7098, 0.7059, 0.7098,  ..., 0.7059, 0.7059, 0.7059],
          ...,
          [0.0784, 0.0745, 0.0667,  ..., 0.0667, 0.0667, 0.0667],
          [0.1451, 0.1412, 0.1373,  ..., 0.0314, 0.0314, 0.0314],
          [0.1804, 0.1765, 0.1725,  ..., 0.0118, 0.0118, 0.0118]],
 
         [[0.6353, 0.6353, 0.6314,  ..., 0.6471, 0.6431, 0.6431],
          [0.6353, 0.6314, 0.6314,  ..., 0.6471, 0.6431, 0.6431],
          [0.6314, 0.6275, 0.63